In [111]:
import pandas as pd
import numpy as np
import ephem
import math
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from dateutil import parser as dateutil_parser
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib
importlib.reload(kinematics)
from kinematics import calc_parallactic_angle, azaltroll_to_theta,apply_mechanical_corrections, q_from_azaltroll, MountModelParams


In [112]:


csv_filename = './n43_SPA_extract.csv'
csv_filename = './n43_MPA1_extract.csv'
csv_filename = './n43_MPA2_extract.csv'
csv_filename = './n115_MPA5_extract.csv'
d = pd.read_csv(csv_filename)
d.describe()
d['dev_p_theta1'] = ((d['s_theta1'] - d['p_theta1'] + 180) % 360 - 180)*60
d['dev_p_theta2'] = ((d['s_theta2'] - d['p_theta2'] + 180) % 360 - 180)*60
d['dev_p_theta3'] = ((d['s_theta3'] - d['p_theta3'] + 180) % 360 - 180)*60
if 'q_theta1' in d.columns:
    d['dev_q_theta1'] = ((d['s_theta1'] - d['q_theta1'] + 180) % 360 - 180)*60
    d['dev_q_theta2'] = ((d['s_theta2'] - d['q_theta2'] + 180) % 360 - 180)*60
    d['dev_q_theta3'] = ((d['s_theta3'] - d['q_theta3'] + 180) % 360 - 180)*60
if 'm_theta1' in d.columns:
    d['dev_m_theta1'] = ((d['s_theta1'] - d['m_theta1'] + 180) % 360 - 180)*60
    d['dev_m_theta2'] = ((d['s_theta2'] - d['m_theta2'] + 180) % 360 - 180)*60
    d['dev_m_theta3'] = ((d['s_theta3'] - d['m_theta3'] + 180) % 360 - 180)*60
d['g_az'] = np.round(d['p_az'] / 5) * 5 
d['g_alt'] = np.round(d['p_alt'] / 5) * 5
d['g_roll'] = np.round(d['p_roll'] / 5) * 5
d.columns

Index(['session_id', 'filename', 'status', 'date_obs', 'ra', 'dec', 'site_lat',
       'site_lon', 'p_az', 'p_alt', 'p_roll', 'p_theta1', 'p_theta2',
       'p_theta3', 's_az', 's_alt', 's_roll', 's_theta1', 's_theta2',
       's_theta3', 'dev_p_az', 'dev_p_alt', 'dev_p_roll', 'pixel_scale_arcsec',
       'dev_p_theta1', 'dev_p_theta2', 'dev_p_theta3', 'g_az', 'g_alt',
       'g_roll'],
      dtype='object')

# Calculate Drift Rates by pairing images with same ra/dec

In [113]:
# Round RA/Dec to 2dp to group true pairs (they should be identical already)
d['ra_key']  = d['ra'].round(2)
d['dec_key'] = d['dec'].round(2)
d['date_obs_dt'] = pd.to_datetime(d['date_obs'])

# Sort so first observation of each pair comes first
d = d.sort_values(['ra_key', 'dec_key', 'date_obs_dt']).reset_index(drop=True)

def compute_residual_drift(grp):
    grp = grp.copy().reset_index(drop=True)

    # Mark all rows as unpaired by default
    grp['pair_seq']         = np.nan
    grp['residual_az_aps']  = np.nan
    grp['residual_alt_aps'] = np.nan
    grp['residual_roll_aps']= np.nan
    grp['dt_sec']           = np.nan

    # Need exactly 2 solved rows with valid s_az/s_alt
    solved = grp[grp['status']=='solved'].dropna(subset=['s_az','s_alt','s_roll'])
    if len(solved) < 2:
        return grp

    # Take first two solved rows as the pair
    img1 = solved.iloc[0]
    img2 = solved.iloc[1]

    dt = (img2['date_obs_dt'] - img1['date_obs_dt']).total_seconds()

    # Sanity check: must be 30-300s apart
    if dt < 30 or dt > 300:
        return grp

    # Sanity check: both images must be near the same sky position
    az1, alt1 = np.radians(img1['s_az']), np.radians(img1['s_alt'])
    az2, alt2 = np.radians(img2['s_az']), np.radians(img2['s_alt'])
    cos_sep = (np.sin(alt1)*np.sin(alt2) +
               np.cos(alt1)*np.cos(alt2)*np.cos(az1-az2))
    sep_deg = np.degrees(np.arccos(np.clip(cos_sep, -1, 1)))
    if sep_deg > 5.0:   # more than 5° apart — wrong pairing
        return grp

    lat_str = str(img1['site_lat'])
    lon_str = str(img1['site_lon'])
    lat_deg = float(img1['site_lat'])

    # ── Step 1: true RA/Dec from image 1 plate solve ──────────────────────
    obs1 = ephem.Observer()
    obs1.lat   = lat_str
    obs1.lon   = lon_str
    obs1.epoch = ephem.J2000
    obs1.date  = ephem.Date(dateutil_parser.parse(img1['date_obs']))
    ra_rad, dec_rad = obs1.radec_of(
        np.radians(img1['s_az']),
        np.radians(img1['s_alt']))

    # ── Step 2: propagate to image 2 timestamp ────────────────────────────
    obs2 = ephem.Observer()
    obs2.lat   = lat_str
    obs2.lon   = lon_str
    obs2.epoch = ephem.J2000
    obs2.date  = ephem.Date(dateutil_parser.parse(img2['date_obs']))
    body = ephem.FixedBody()
    body._ra    = ra_rad
    body._dec   = dec_rad
    body._epoch = ephem.J2000
    body.compute(obs2)

    exp_az2  = np.degrees(float(body.az))
    exp_alt2 = np.degrees(float(body.alt))

    # ── Step 3: expected roll change ──────────────────────────────────────
    def pa(az_deg, alt_deg, lat_deg):
        if abs(alt_deg - 90.0) < 1e-6:
            return 0.0
        az  = math.radians(az_deg)
        alt = math.radians(alt_deg)
        lat = math.radians(lat_deg)
        num = math.sin(az)
        den = math.tan(lat)*math.cos(alt) - math.sin(alt)*math.cos(az)
        return ((- math.degrees(math.atan2(num, den)) + 180) % 360) - 180

    pa1 = pa(img1['s_az'], img1['s_alt'], lat_deg)
    pa2 = pa(exp_az2,      exp_alt2,      lat_deg)
    expected_roll2 = ((img1['s_roll'] + (pa2 - pa1) + 180) % 360) - 180

    # ── Step 4: residuals ─────────────────────────────────────────────────
    res_az   = ((img2['s_az']   - exp_az2         + 180) % 360 - 180)
    res_alt  =   img2['s_alt']  - exp_alt2
    res_roll = ((img2['s_roll'] - expected_roll2   + 180) % 360 - 180)

    # Write results back to the rows by index
    grp.loc[solved.index[0], 'pair_seq']          = 0
    grp.loc[solved.index[1], 'pair_seq']          = 1
    grp.loc[solved.index[0], 'dt_sec']            = dt
    grp.loc[solved.index[0], 'solved_ra_deg']     = np.degrees(float(ra_rad))
    grp.loc[solved.index[0], 'solved_dec_deg']    = np.degrees(float(dec_rad))
    grp.loc[solved.index[0], 'expected_az2']      = exp_az2
    grp.loc[solved.index[0], 'expected_alt2']     = exp_alt2
    grp.loc[solved.index[0], 'expected_roll2']    = expected_roll2
    grp.loc[solved.index[0], 'residual_az_aps']   = (res_az   * 3600) / dt
    grp.loc[solved.index[0], 'residual_alt_aps']  = (res_alt  * 3600) / dt
    grp.loc[solved.index[0], 'residual_roll_aps'] = (res_roll * 3600) / dt

    return grp

d = d.groupby(['ra_key','dec_key'], group_keys=False).apply(compute_residual_drift)

# Only keep valid pair_seq==0 rows — both images were solved and pair is geometrically valid
pairs = d[d['pair_seq']==0].dropna(subset=['residual_az_aps']).copy()
pairs['mid_az']  = pairs['p_az']
pairs['mid_alt'] = pairs['p_alt']

print(f"Valid pairs: {len(pairs)}")
print()
print("RESIDUAL drift after removing expected sidereal motion:")
for col, label in [('residual_az_aps','az'),
                   ('residual_alt_aps','alt'),
                   ('residual_roll_aps','roll')]:
    v = pairs[col]
    print(f"  {label:6s}  mean={v.mean():+.3f}  std={v.std():.3f}  "
          f"min={v.min():+.3f}  max={v.max():+.3f}  arcsec/sec")

Valid pairs: 50

RESIDUAL drift after removing expected sidereal motion:
  az      mean=+0.499  std=2.539  min=-5.667  max=+9.455  arcsec/sec
  alt     mean=+0.977  std=2.408  min=-5.269  max=+5.478  arcsec/sec
  roll    mean=+1.203  std=29.773  min=-56.662  max=+56.514  arcsec/sec


C:\Users\Nina\AppData\Local\Temp\ipykernel_4568\3909428608.py:107: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



# Sky Space Sidereal Tracking Drift - Scatter Plots

In [114]:
fig = px.scatter_matrix(pairs, dimensions=[ "p_az", "p_alt", "p_roll", "residual_az_aps", "residual_alt_aps", "residual_roll_aps"], color="p_alt",
                        hover_data=[ "p_az", "p_alt", "p_roll", "filename",])
fig.update_layout(height=800,width=800 )
fig.show()

# Polar Plot of Az/Alt Sidereal Residual Drift Rates

In [115]:
import numpy as np
import plotly.graph_objects as go

def sky_polar(pairs, px_per_as=3.0, title="Night sky drift vectors"):
    """
    Polar sky plot with true drift vectors.

    Vector decomposition in screen space:
      az drift  → tangential direction (perpendicular to radial, +East)
      alt drift → radial direction (+alt = inward toward zenith)

    px_per_as : pixels per arcsec/s. Tune until arrows are readable.
    """
    def to_xy(az_deg, alt_deg):
        r = 1.0 - alt_deg / 90.0
        a = np.radians(az_deg - 90)   # N up
        return r * np.cos(a), r * np.sin(a)

    fig = go.Figure()

    # ── Grid rings ────────────────────────────────────────────────────────
    theta = np.linspace(0, 2*np.pi, 360)
    for alt in [30, 60]:
        r = 1.0 - alt/90.0
        fig.add_trace(go.Scatter(
            x=r*np.cos(theta), y=r*np.sin(theta),
            mode='lines', line=dict(color='rgba(150,150,220,0.2)', width=0.8, dash='dot'),
            hoverinfo='skip', showlegend=False))
        fig.add_annotation(x=0.02, y=-(r)+0.04, text=f"{alt}°",
            showarrow=False, font=dict(size=10, color='rgba(180,180,220,0.6)'))

    # Horizon
    fig.add_trace(go.Scatter(
        x=np.cos(theta), y=np.sin(theta),
        mode='lines', line=dict(color='rgba(150,150,220,0.3)', width=1.2),
        hoverinfo='skip', showlegend=False))

    # Compass spokes + labels
    for label, az in zip(['N','E','S','W'], [0, 90, 180, 270]):
        x0, y0 = to_xy(az, 0)
        fig.add_shape(type='line', x0=0, y0=0, x1=x0, y1=y0,
            line=dict(color='rgba(150,150,220,0.2)', width=0.7, dash='dot'))
        lx, ly = to_xy(az, -5)
        fig.add_annotation(x=lx*1.08, y=ly*1.08, text=f"<b>{label}</b>",
            showarrow=False, font=dict(size=14, color='rgba(200,210,255,0.75)'))

    # ── Drift vectors ─────────────────────────────────────────────────────
    # Scale: convert arcsec/s to unit-circle displacement
    # 1.0 unit = 90° of altitude. So 1 arcsec/s in alt = (1/90)/3600 units/s... 
    # Instead we scale directly in figure units for readability.
    # Empirically: max drift ~19"/s, map to ~0.25 units → scale = 0.25/19
    max_drift = np.sqrt(pairs['residual_az_aps']**2 + pairs['residual_alt_aps']**2).max()
    scale = 0.22 / max_drift   # tune 0.22 to taste

    color_map = lambda m: '#1D9E75' if m < 5 else '#EF9F27' if m < 10 else '#D85A30'

    for _, row in pairs.iterrows():
        az0 = row['mid_az']
        alt0 = row['mid_alt']
        daz  = row['residual_az_aps']
        dalt = row['residual_alt_aps']
        mag  = np.sqrt(daz**2 + dalt**2)
        color = color_map(mag)

        x0, y0 = to_xy(az0, alt0)

        # Unit vectors in figure space
        az_rad = np.radians(az0 - 90)
        tan_x =  np.sin(az_rad)   # tangential (+East / clockwise)
        tan_y = -np.cos(az_rad)
        rad_x = -np.cos(az_rad)   # radial inward (+alt = toward zenith)
        rad_y = -np.sin(az_rad)

        vx = daz * scale * tan_x + dalt * scale * rad_x
        vy = daz * scale * tan_y + dalt * scale * rad_y

        tooltip = (f"Az={az0:.1f}°  Alt={alt0:.1f}°<br>"
                   f"drift az={daz:+.2f}\"/s  alt={dalt:+.2f}\"/s<br>"
                   f"magnitude={mag:.2f}\"/s")

        # Origin dot
        fig.add_trace(go.Scatter(
            x=[x0], y=[y0], mode='markers',
            marker=dict(size=9, color=color, line=dict(width=1, color='#0d1117')),
            hovertemplate=tooltip+'<extra></extra>', showlegend=False))

        # Arrow
        fig.add_annotation(
            x=x0+vx, y=y0+vy, ax=x0, ay=y0,
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=2, arrowsize=1.3,
            arrowwidth=2.2, arrowcolor=color)

    # ── Legend ────────────────────────────────────────────────────────────
    for label, color in [('< 5"/s', '#1D9E75'),
                          ('5–10"/s','#EF9F27'),
                          ('> 10"/s','#D85A30')]:
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(size=10, color=color), name=label))

    # Scale reference bar (bottom-left)
    ref_as = 10.0
    ref_len = ref_as * scale
    fig.add_annotation(
        x=-0.95 + ref_len, y=-0.92, ax=-0.95, ay=-0.92,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowwidth=2,
        arrowcolor='rgba(180,180,220,0.6)')
    fig.add_annotation(x=-0.95, y=-0.87, text='10"/s ref',
        showarrow=False, font=dict(size=10, color='rgba(180,180,220,0.6)'))

    # Zenith dot
    fig.add_trace(go.Scatter(x=[0], y=[0], mode='markers',
        marker=dict(size=5, color='white', opacity=0.4),
        hoverinfo='skip', showlegend=False))

    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor='center',
                   font=dict(size=14, color='rgba(200,210,255,0.9)')),
        xaxis=dict(range=[-1.18, 1.18], showgrid=False, zeroline=False,
                   showticklabels=False, scaleanchor='y'),
        yaxis=dict(range=[-1.18, 1.18], showgrid=False, zeroline=False,
                   showticklabels=False),
        plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
        font=dict(color='#aaaacc'),
        legend=dict(x=0.01, y=0.01, bgcolor='rgba(0,0,0,0.35)',
                    bordercolor='rgba(200,200,255,0.2)', borderwidth=1),
        width=660, height=640,
        margin=dict(l=20, r=20, t=50, b=20),
    )
    return fig

fig = sky_polar(pairs)
fig.show()

In [116]:
az  = np.radians(pairs['mid_az'].values)
alt = np.radians(pairs['mid_alt'].values)

predictors = {
    'cos(az)':          np.cos(az),
    'sin(az)':          np.sin(az),
    'cos(az)*tan(alt)': np.cos(az)*np.tan(alt),
    'sin(az)*tan(alt)': np.sin(az)*np.tan(alt),
    'tan(alt)':         np.tan(alt),
    'sin(alt)':         np.sin(alt),
}

print(f"{'predictor':25s}  {'r(res_az)':>10s}  {'r(res_alt)':>10s}  {'r(res_roll)':>11s}")
print("-" * 62)
for pname, pval in predictors.items():
    r_az   = np.corrcoef(pval, pairs['residual_az_aps'])[0,1]
    r_alt  = np.corrcoef(pval, pairs['residual_alt_aps'])[0,1]
    r_roll = np.corrcoef(pval, pairs['residual_roll_aps'])[0,1]
    print(f"{pname:25s}  {r_az:>+10.3f}  {r_alt:>+10.3f}  {r_roll:>+11.3f}")

print()

# Fit the polar model to the clean residuals
from numpy.linalg import lstsq

tan_alt = np.tan(alt)
A_alt = np.column_stack([ np.cos(az),             np.sin(az)           ])
A_az  = np.column_stack([-np.sin(az) * tan_alt,   np.cos(az) * tan_alt ])
A_full = np.vstack([A_alt, A_az])
b_full = np.concatenate([pairs['residual_alt_aps'].values,
                          pairs['residual_az_aps'].values])

coeffs,_,_,_ = lstsq(A_full, b_full, rcond=None)
dAz_pole, dAlt_pole = coeffs

OMEGA = 15.0411
RAD_TO_ARCMIN = np.degrees(1) * 60
pole_az_arcmin  = (dAz_pole  / OMEGA) * RAD_TO_ARCMIN
pole_alt_arcmin = (dAlt_pole / OMEGA) * RAD_TO_ARCMIN
pole_total      = np.sqrt(pole_az_arcmin**2 + pole_alt_arcmin**2)
pole_dir        = np.degrees(np.arctan2(pole_az_arcmin, pole_alt_arcmin)) % 360

def r2(obs, pred):
    ss_r = np.sum((obs-pred)**2)
    ss_t = np.sum((obs-obs.mean())**2)
    return 1 - ss_r/ss_t if ss_t > 0 else float('nan')

r2_alt = r2(pairs['residual_alt_aps'].values, A_alt @ coeffs)
r2_az  = r2(pairs['residual_az_aps'].values,  A_az  @ coeffs)

print(f"Polar fit on RESIDUALS:")
print(f"  dAz_pole  = {dAz_pole:+.4f} \"/s   R²(alt)={r2_alt:.3f}")
print(f"  dAlt_pole = {dAlt_pole:+.4f} \"/s   R²(az) ={r2_az:.3f}")
print(f"  Pole offset: {pole_az_arcmin:+.2f}' az,  {pole_alt_arcmin:+.2f}' alt")
print(f"  Total: {pole_total:.2f}'  toward az={pole_dir:.1f}°")
print()

# And for roll — what's the dominant predictor?
print("Roll residual fit vs cos(az):")
A_roll = np.column_stack([np.cos(az), np.sin(az), np.ones(len(az))])
c_roll,_,_,_ = lstsq(A_roll, pairs['residual_roll_aps'].values, rcond=None)
pred_roll = A_roll @ c_roll
r2_roll = r2(pairs['residual_roll_aps'].values, pred_roll)
print(f"  cos(az) coeff = {c_roll[0]:+.4f}  sin(az) coeff = {c_roll[1]:+.4f}  "
      f"const = {c_roll[2]:+.4f}   R²={r2_roll:.3f}")
print(f"  Amplitude = {np.sqrt(c_roll[0]**2+c_roll[1]**2):.4f} arcsec/sec")

predictor                   r(res_az)  r(res_alt)  r(res_roll)
--------------------------------------------------------------
cos(az)                        +0.403      +0.041       +0.965
sin(az)                        +0.403      -0.634       -0.064
cos(az)*tan(alt)               +0.484      +0.059       +0.976
sin(az)*tan(alt)               +0.357      -0.595       -0.073
tan(alt)                       +0.324      +0.368       -0.092
sin(alt)                       +0.273      +0.447       -0.112

Polar fit on RESIDUALS:
  dAz_pole  = -0.5431 "/s   R²(alt)=-0.272
  dAlt_pole = +0.1436 "/s   R²(az) =0.121
  Pole offset: -124.14' az,  +32.82' alt
  Total: 128.40'  toward az=284.8°

Roll residual fit vs cos(az):
  cos(az) coeff = +39.6383  sin(az) coeff = -1.6253  const = -1.2161   R²=0.932
  Amplitude = 39.6716 arcsec/sec
